# Q6 — A divergence measure for need and demand (jump board)

Rebuilds Brian's Q1 county panel from the raw Food Group data (`org_FSStats.xlsx`)
and Map the Meal Gap, applying his cleaning decisions (Decisions 1, 2, 3, 6 and 8)
and county/FIPS mapping rather than inventing new ones. The checkpoints match his
pipeline exactly (18,692 rows after dedup, 98 impossible rows, 258 county-years
across 86 counties). See `cleaning_log.md` for the full provenance (Decisions 1–8,
the sensitivity test, the Steele County finding).

**Q6 asks for three things, not one:** (1) build a need metric that is more than
demand, (2) compare it to the three food-shelf metrics and find where they
diverge, (3) use county characteristics to explain the divergence, "akin to
Question 1." The graduate addendum on top of that asks for a *falsifiable*
definition: what would make us reject or revise this measure.


In [20]:
import pandas as pd, numpy as np
from scipy import stats
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

ROOT = Path('/content/drive/MyDrive/MinneMUDAC 26 for BU')
DATA = ROOT / 'By Organizers' / 'Data'   # the organizers' original files

# Pinned paths: the shared Drive holds several copies of these files (teammates'
# folders), and a search would silently pick whichever copy it finds first.
PATHS = {
    'org_FSStats.xlsx':             DATA / 'org_FSStats.xlsx',
    'Organization.xlsx':            DATA / 'Organization.xlsx',
    'marketbucksUsage.xlsx':        DATA / 'marketbucksUsage.xlsx',
    'Incoming Call.xlsx':           DATA / 'Incoming Call.xlsx',
    'MMG_County_MN_2021_2024.xlsx': ROOT / 'MMG_County_MN_2021_2024.xlsx',
}

def find_and_load(filename):
    path = PATHS[filename]
    if not path.exists():
        # fall back to a search, but say so loudly
        hits = list(ROOT.rglob(filename))
        if not hits:
            raise FileNotFoundError(f"Couldn't find {filename!r} anywhere under {ROOT}")
        print(f"WARNING: {path} not found; falling back to {hits[0]}")
        path = hits[0]
    print(f"{filename} -> {path}")
    return pd.read_excel(path)

fs    = find_and_load('org_FSStats.xlsx')
orgs  = find_and_load('Organization.xlsx')
mb    = find_and_load('marketbucksUsage.xlsx')
calls = find_and_load('Incoming Call.xlsx')
mmg   = find_and_load('MMG_County_MN_2021_2024.xlsx')

fs.shape, orgs.shape, mb.shape, calls.shape, mmg.shape
# expect: (28287, 58) (1202, 68) (13573, 19) (49727, 31) (348, 26)

Mounted at /content/drive
org_FSStats.xlsx -> /content/drive/MyDrive/MinneMUDAC 26 for BU/By Organizers/Data/org_FSStats.xlsx
Organization.xlsx -> /content/drive/MyDrive/MinneMUDAC 26 for BU/By Organizers/Data/Organization.xlsx
marketbucksUsage.xlsx -> /content/drive/MyDrive/MinneMUDAC 26 for BU/By Organizers/Data/marketbucksUsage.xlsx
Incoming Call.xlsx -> /content/drive/MyDrive/MinneMUDAC 26 for BU/By Organizers/Data/Incoming Call.xlsx
MMG_County_MN_2021_2024.xlsx -> /content/drive/MyDrive/MinneMUDAC 26 for BU/MMG_County_MN_2021_2024.xlsx


((28287, 58), (1202, 68), (13573, 19), (49727, 31), (348, 26))

In [21]:
# Decision 1 — DateForSearch is the date of record; drop _Year/_Month
empty = fs.columns[fs.notna().sum() == 0]
fs = fs.drop(columns=empty)
fs = fs.drop(columns=['SuperShelf'])
fs = fs.drop(columns=['_Year', '_Month'])

mmg.columns = [str(c).strip() for c in mmg.columns]
mmg['fips5'] = mmg.FIPS.astype(str).str.zfill(5)

M = ['HouseholdsReg', 'IndividualsReg', 'PoundsReg']
fsq1 = fs[fs.DateForSearch.dt.year.isin([2022, 2023, 2024])].copy()
print("after window filter:", fsq1.shape)

# Decision 2 — duplicate (site, month) keys: keep the larger row
fsq1 = (fsq1
    .sort_values(['_id_site', 'DateForSearch', 'IndividualsReg', 'PoundsReg', '__ID'],
                 ascending=[True, True, False, False, False])
    .drop_duplicates(['_id_site', 'DateForSearch'], keep='first')
    .copy())

bad = fsq1.HouseholdsReg > fsq1.IndividualsReg
print("after dedup:", fsq1.shape, "| impossible rows:", bad.sum())

# Decision 3 — HouseholdsReg > IndividualsReg: 98 impossible rows
mask = (fsq1._id_site == 1197.0) & (fsq1.DateForSearch == '2023-07-01')
fsq1['hh_imputed'] = False
fsq1.loc[mask, 'HouseholdsReg'] = 311
fsq1.loc[mask, 'hh_imputed']    = True

def ratio(df, key):
    g = df.groupby(key)[['IndividualsReg', 'HouseholdsReg']].sum()
    r = g.IndividualsReg / g.HouseholdsReg
    return r.where(r >= 1.5)

clean    = fsq1[~bad]
site_r   = ratio(clean, '_id_site')
county_r = ratio(clean, 'County')
state_r  = clean.IndividualsReg.sum() / clean.HouseholdsReg.sum()

fixable = bad & (fsq1.IndividualsReg > 0) & ~fsq1.hh_imputed
r = (fsq1._id_site.map(site_r)
         .fillna(fsq1.County.map(county_r))
         .fillna(state_r))

fsq1['hh_adjusted'] = bad
fsq1.loc[fixable, 'HouseholdsReg'] = (fsq1.loc[fixable, 'IndividualsReg'] / r[fixable]).round()
print("hh_adjusted:", fsq1.hh_adjusted.sum(), "| hh_imputed:", fsq1.hh_imputed.sum(), "| fixable:", fixable.sum())

# Decision 8 — county label standardization + FIPS map
FIX = {'Hennepin-Minneapolis': 'Hennepin', 'Hennepin-Suburbs': 'Hennepin',
       'Ramsey-Saint Paul':    'Ramsey',   'Ramsey-Suburbs':   'Ramsey',
       'Lac Qui Parle':        'Lac qui Parle',
       'Saint Louis':          'St. Louis'}

fsq1['county_detail'] = fsq1.County
fsq1['county']        = fsq1.County.replace(FIX)

fips = (mmg.assign(c=mmg['County, State'].str.replace(' County, Minnesota', '', regex=False))
            .drop_duplicates('c').set_index('c').fips5)
fsq1['county_fips'] = fsq1.county.map(fips)
print("distinct counties after standardization:", fsq1.county.nunique())

# Decision 6 — recover 3 sites, carry the rest as a named residual
RECOVER = {15362.0: 'Rice', 15363.0: 'Rice', 15218.0: 'Cass'}
fsq1['county']            = fsq1.county.fillna(fsq1._id_site.map(RECOVER))
fsq1['county_fips']       = fsq1.county.map(fips)
fsq1['county_unassigned'] = fsq1.county.isna()
fsq1['year']              = fsq1.DateForSearch.dt.year
print("county_unassigned rows:", fsq1.county_unassigned.sum())

# Build the county-year panel
county_year = (fsq1[~fsq1.county_unassigned]
               .groupby(['county', 'county_fips', 'year'], as_index=False)[M].sum())
unassigned  = fsq1[fsq1.county_unassigned].groupby('year')[M].sum()

for c in M:
    a, b, t = county_year[c].sum(), unassigned[c].sum(), fsq1[c].sum()
    assert np.isclose(a + b, t), c

print("county_year rows, distinct counties:", len(county_year), county_year.county.nunique())

after window filter: (18756, 31)
after dedup: (18692, 31) | impossible rows: 98
hh_adjusted: 98 | hh_imputed: 1 | fixable: 93
distinct counties after standardization: 86
county_unassigned rows: 90
county_year rows, distinct counties: 258 86


In [22]:
mmg_s = (mmg.assign(county=mmg['County, State'].str.replace(' County, Minnesota', '', regex=False))
            .rename(columns={
                'Year': 'year',
                '# of Food Insecure Persons Overall': 'fi_persons',
                'Overall Food Insecurity Rate': 'fi_rate',
                'Cost Per Meal': 'cost_per_meal',
                '% FI ≤ SNAP Threshold': 'fi_below_snap',
                'Rural-Urban Continuum Code (2023)': 'rucc',
                'Child Food Insecurity Rate': 'child_fi_rate',
            })
            [['county', 'year', 'fi_persons', 'fi_rate', 'cost_per_meal',
              'fi_below_snap', 'rucc', 'child_fi_rate']])

panel = county_year.merge(mmg_s, on=['county', 'year'])
panel['population']        = panel.fi_persons / panel.fi_rate
panel['visits_per1k']      = panel.HouseholdsReg  / panel.population * 1000
panel['individuals_per1k'] = panel.IndividualsReg / panel.population * 1000
panel['pounds_per1k']      = panel.PoundsReg      / panel.population * 1000
panel.shape

(258, 16)

## 1. The need index

Three components, all from Map the Meal Gap and already in Brian's Q1 panel:

| Component | What it captures | Why it's more than `fi_rate` alone |
|---|---|---|
| `fi_rate` | MMG food-insecurity rate | prevalence |
| `cost_per_meal` | MMG local meal-cost estimate | how hard need is to self-resolve |
| `fi_below_snap` | % of food-insecure persons at/below the SNAP income threshold | depth/severity of poverty among the food-insecure, not just whether they're insecure |

`child_fi_rate` and `rucc` (rurality) are deliberately **left out of the index**
— they're either redundant with `fi_rate` (r=0.91 for child rate) or describe
*access*, not underlying need. They're kept as explanatory variables instead,
the same role county characteristics played in Q1.


In [23]:
NEED_COLS = ["fi_rate", "cost_per_meal", "fi_below_snap"]

def zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

for c in NEED_COLS:
    panel[f"z_{c}"] = panel.groupby("year")[c].transform(zscore)   # within-year: MMG is cross-sectional only

panel["need_index"] = panel[[f"z_{c}" for c in NEED_COLS]].mean(axis=1)
panel[["county","year"] + NEED_COLS + ["need_index"]].sort_values("need_index", ascending=False).head(10)

,county,year,fi_rate,cost_per_meal,fi_below_snap,need_index
126,Mahnomen,2022,0.157,4.17,0.955,2.050299
127,Mahnomen,2023,0.165,3.67,0.899,1.941658
128,Mahnomen,2024,0.167,3.79,0.863,1.508617
134,Martin,2024,0.154,3.95,0.826,1.337239
227,Swift,2024,0.157,3.72,0.808,1.064854
31,Cass,2023,0.126,4.04,0.759,1.054745
32,Cass,2024,0.132,4.18,0.774,1.037875
0,Aitkin,2022,0.130,4.37,0.754,1.030725
30,Cass,2022,0.115,4.37,0.811,0.912332
2,Aitkin,2024,0.150,3.86,0.740,0.866211


## 2. The demand side

Reuse Brian's Q1 rate metrics (`visits_per1k`, `individuals_per1k`,
`pounds_per1k`), built in the panel cell above and already population-adjusted.


In [24]:
DEMAND_COLS = ["visits_per1k", "individuals_per1k", "pounds_per1k"]
for c in DEMAND_COLS:
    panel[f"z_{c}"] = panel.groupby("year")[c].transform(zscore)
panel["demand_index"] = panel[[f"z_{c}" for c in DEMAND_COLS]].mean(axis=1)
panel["divergence"] = panel["demand_index"] - panel["need_index"]
# positive -> activity exceeds what need predicts (over-served / easy-access draw)
# negative -> need exceeds activity (the barrier case the brief describes)
panel[["county","year","need_index","demand_index","divergence"]].sort_values("divergence").head(10)

,county,year,need_index,demand_index,divergence
227,Swift,2024,1.064854,-0.759008,-1.823861
75,Grant,2022,0.580074,-1.035673,-1.615747
12,Benton,2022,0.414372,-1.143481,-1.557853
225,Swift,2022,0.754603,-0.775904,-1.530507
226,Swift,2023,0.766723,-0.756640,-1.523363
157,Nobles,2023,0.321580,-1.121448,-1.443027
134,Martin,2024,1.337239,-0.093367,-1.430606
2,Aitkin,2024,0.866211,-0.515151,-1.381362
228,Todd,2022,0.525507,-0.835837,-1.361343
0,Aitkin,2022,1.030725,-0.317840,-1.348564


### 2b. Need against each of the three food-shelf metrics

The brief asks us to compare the new measure against **each** of the three food-shelf metrics, not only a composite. Two checks:
1. How strongly does `need_index` track each metric, year by year, and is it a better or worse match than `fi_rate` alone?
2. Do the three metrics agree with each other? If they don't, averaging them into `demand_index` would hide real differences.

In [25]:
# 1. correlation of need (and fi_rate, for comparison) with each metric, within each year
rows = []
for yr, g in panel.groupby("year"):
    for c in DEMAND_COLS:
        rows.append({"year": yr, "metric": c,
                     "r_need_index": g["need_index"].corr(g[c]),
                     "r_fi_rate":    g["fi_rate"].corr(g[c])})
metric_corr = pd.DataFrame(rows)
print("Correlation with each food-shelf metric, by year:")
print(metric_corr.pivot(index="metric", columns="year", values=["r_need_index", "r_fi_rate"]).round(3))

# 2. do the three metrics agree with each other? (within-year z-scores, pooled)
print("\nAgreement among the three metrics (correlation of within-year z-scores):")
print(panel[[f"z_{c}" for c in DEMAND_COLS]].corr().round(3))

Correlation with each food-shelf metric, by year:
                  r_need_index               r_fi_rate              
year                      2022   2023   2024      2022   2023   2024
metric                                                              
individuals_per1k        0.125  0.007 -0.103     0.076 -0.034 -0.150
pounds_per1k             0.290  0.208 -0.009     0.315  0.197 -0.029
visits_per1k             0.190  0.079 -0.045     0.137  0.053 -0.113

Agreement among the three metrics (correlation of within-year z-scores):
                     z_visits_per1k  z_individuals_per1k  z_pounds_per1k
z_visits_per1k                1.000                0.961           0.864
z_individuals_per1k           0.961                1.000           0.854
z_pounds_per1k                0.864                0.854           1.000


## 3. Statistical outliers — does the richer index change Q1's answer?

Q1 flagged off-trend counties by regressing each demand rate on `fi_rate` alone
(`resid_t_*_rate`, |t| > 2). Re-run the identical procedure, swapping in
`need_index`, and compare the two outlier lists.


In [26]:
def residual_t(df, y_col, x_col):
    slope, intercept, r, p, se = stats.linregress(df[x_col], df[y_col])
    resid = df[y_col] - (intercept + slope * df[x_col])
    return resid / resid.std(ddof=2)

# -- need_index version (original) --
for c in DEMAND_COLS:
    panel[f"needidx_resid_t_{c}"] = panel.groupby("year", group_keys=False).apply(
        lambda g: residual_t(g, c, "need_index"), include_groups=False)
panel["needidx_offtrend_any"] = (panel[[f"needidx_resid_t_{c}" for c in DEMAND_COLS]].abs() > 2).any(axis=1)

# -- fi_rate version (Q1's original method, re-run here so it can be compared) --
for c in DEMAND_COLS:
    panel[f"resid_t_{c}"] = panel.groupby("year", group_keys=False).apply(
        lambda g: residual_t(g, c, "fi_rate"), include_groups=False)
panel["offtrend_any_rate"] = (panel[[f"resid_t_{c}" for c in DEMAND_COLS]].abs() > 2).any(axis=1)

compare = panel.groupby("county").agg(
    years_offtrend_q1=("offtrend_any_rate", "sum"),
    years_offtrend_needidx=("needidx_offtrend_any", "sum"),
    mean_divergence=("divergence", "mean"),
).reset_index()

print("Newly flagged (Q1 missed, richer index catches):")
print(compare[(compare.years_offtrend_q1==0) & (compare.years_offtrend_needidx>0)])
print("\nCleared (Q1 flagged on fi_rate alone, richer index clears):")
print(compare[(compare.years_offtrend_q1>0) & (compare.years_offtrend_needidx==0)])
print("\nPersistent under BOTH (strongest findings):")
print(compare[(compare.years_offtrend_q1==3) & (compare.years_offtrend_needidx==3)].sort_values('mean_divergence'))

Newly flagged (Q1 missed, richer index catches):
   county  years_offtrend_q1  years_offtrend_needidx  mean_divergence
15   Cook                  0                       1         1.259718

Cleared (Q1 flagged on fi_rate alone, richer index clears):
    county  years_offtrend_q1  years_offtrend_needidx  mean_divergence
61  Ramsey                  1                       0         0.197469

Persistent under BOTH (strongest findings):
        county  years_offtrend_q1  years_offtrend_needidx  mean_divergence
65        Rice                  3                       3         3.148400
73      Steele                  3                       3         3.594843
81  Washington                  3                       3         3.702741


In [27]:
r_need_fi = panel[["need_index", "fi_rate"]].corr().iloc[0, 1]
print(f"corr(need_index, fi_rate) = {r_need_fi:.3f}")

corr(need_index, fi_rate) = 0.723


In [28]:
print(f"shared variance (r^2) = {r_need_fi**2:.3f}")
print(panel[["fi_rate", "cost_per_meal", "fi_below_snap"]].corr().round(3))

shared variance (r^2) = 0.523
               fi_rate  cost_per_meal  fi_below_snap
fi_rate          1.000         -0.551          0.578
cost_per_meal   -0.551          1.000         -0.351
fi_below_snap    0.578         -0.351          1.000


**Result:** the statistical outlier test barely moves. `need_index` is highly correlated with `fi_rate` (r = 0.723, computed above), so the strict |t| > 2 test mostly reproduces Q1's own over-served list: **Rice, Steele, Washington** are flagged in all 3 years under both methods. Steele is Brian's regional-catchment finding from Q1 (see the cleaning log). The richer index changes only two counties: **Cook** is newly flagged (mean divergence = +1.26, over-served direction) and **Ramsey** drops off.

**That's a real but thin result for Q6 on its own.** The raw divergence ranking is where the interesting story is, not the statistical threshold. The under-served direction never gets large enough in any single year to cross |t| > 2, but the *same counties* show up at the bottom of the ranking every year.

In [29]:
ranked = panel.copy()
ranked["rank_divergence"] = ranked.groupby("year")["divergence"].rank()
bottom10 = ranked[ranked.rank_divergence <= 10]
persistent_underserved = bottom10.groupby("county").size().sort_values(ascending=False)
persistent_underserved.head(10)

,0
county,
Aitkin,3
Benton,3
Grant,3
Todd,3
Norman,3
Nicollet,3
Swift,3
Wadena,2
Chippewa,2


## 4. The finding: a persistent rural, under-served cluster

**Swift, Grant, Todd, Benton, Aitkin, Norman, Nicollet** rank in the bottom 10 (most under-served) in all 3 years, and **Wadena** and **Chippewa** in 2 of 3. (Becker appears once, but it has under 35% household-figure coverage, so we leave it out; see caveats.) Look at what they share:


In [30]:
# --- build shelves / top_site_share / shelves_per_10k_fi from fsq1 (site-level, post-cleaning) ---

active = fsq1[~fsq1.county_unassigned]

site_counts = (active.groupby(['county', 'year'])['_id_site']
               .nunique()
               .rename('shelves')
               .reset_index())

site_totals = active.groupby(['county', 'year', '_id_site'])['IndividualsReg'].sum().reset_index()
county_totals = site_totals.groupby(['county', 'year'])['IndividualsReg'].sum()
top_site = site_totals.groupby(['county', 'year'])['IndividualsReg'].max()
top_site_share = (top_site / county_totals).rename('top_site_share').reset_index()

# drop any columns from a previous run so re-running this cell doesn't create _x/_y duplicates
panel = panel.drop(columns=["shelves", "sites_reporting", "top_site_share", "shelves_per_10k_fi"], errors="ignore")

panel = panel.merge(site_counts, on=['county', 'year'], how='left')
panel = panel.merge(top_site_share, on=['county', 'year'], how='left')
panel["shelves_per_10k_fi"] = panel.shelves / panel.fi_persons * 10_000

print(panel.columns.tolist())  # sanity check: shelves, top_site_share, shelves_per_10k_fi should be in here

['county', 'county_fips', 'year', 'HouseholdsReg', 'IndividualsReg', 'PoundsReg', 'fi_persons', 'fi_rate', 'cost_per_meal', 'fi_below_snap', 'rucc', 'child_fi_rate', 'population', 'visits_per1k', 'individuals_per1k', 'pounds_per1k', 'z_fi_rate', 'z_cost_per_meal', 'z_fi_below_snap', 'need_index', 'z_visits_per1k', 'z_individuals_per1k', 'z_pounds_per1k', 'demand_index', 'divergence', 'needidx_resid_t_visits_per1k', 'needidx_resid_t_individuals_per1k', 'needidx_resid_t_pounds_per1k', 'needidx_offtrend_any', 'resid_t_visits_per1k', 'resid_t_individuals_per1k', 'resid_t_pounds_per1k', 'offtrend_any_rate', 'shelves', 'top_site_share', 'shelves_per_10k_fi']


In [31]:
underserved_cluster = ["Swift","Grant","Todd","Benton","Aitkin","Norman","Nicollet","Wadena","Chippewa"]
cols = ["county","year","divergence","rucc","fi_below_snap","shelves","shelves_per_10k_fi","top_site_share"]
panel[panel.county.isin(underserved_cluster)][cols].sort_values(["county", "year"])

,county,year,divergence,rucc,fi_below_snap,shelves,shelves_per_10k_fi,top_site_share
0,Aitkin,2022,-1.348564,8.0,0.754,9,43.689320,0.289516
1,Aitkin,2023,-1.141858,8.0,0.706,8,36.529680,0.294052
2,Aitkin,2024,-1.381362,8.0,0.740,6,24.793388,0.470404
12,Benton,2022,-1.557853,3.0,0.682,3,7.246377,0.617321
13,Benton,2023,-1.285700,3.0,0.630,3,6.276151,0.593328
14,Benton,2024,-1.094861,3.0,0.600,3,5.714286,0.662485
33,Chippewa,2022,-1.024432,7.0,0.778,1,7.092199,1.000000
34,Chippewa,2023,-1.140984,7.0,0.687,1,6.211180,1.000000
35,Chippewa,2024,-1.022845,7.0,0.685,1,5.524862,1.000000
75,Grant,2022,-1.615747,9.0,0.840,2,31.746032,1.000000


In [32]:
overserved_cluster = ["Rice", "Steele", "Washington"]
panel[panel.county.isin(overserved_cluster)][cols].sort_values(["county", "year"])

,county,year,divergence,rucc,fi_below_snap,shelves,shelves_per_10k_fi,top_site_share
195,Rice,2022,1.930469,4.0,0.591,5,8.695652,0.485087
196,Rice,2023,3.498649,4.0,0.526,7,10.638298,0.373614
197,Rice,2024,4.016081,4.0,0.527,7,9.345794,0.325844
219,Steele,2022,2.523335,5.0,0.697,3,9.118541,0.915063
220,Steele,2023,3.453750,5.0,0.636,3,8.498584,0.934545
221,Steele,2024,4.807445,5.0,0.624,3,7.317073,0.952089
243,Washington,2022,3.361396,1.0,0.471,13,7.475561,0.775664
244,Washington,2023,3.942372,1.0,0.432,13,6.187530,0.718829
245,Washington,2024,3.804456,1.0,0.436,13,5.782918,0.768698


In [33]:
# divergence on each metric separately: do both clusters hold on all three metrics,
# or only on the composite? (county means, 2022-2024; negative = need exceeds activity)
for c in DEMAND_COLS:
    panel[f"div_{c}"] = panel[f"z_{c}"] - panel["need_index"]
div_cols = [f"div_{c}" for c in DEMAND_COLS]

(panel[panel.county.isin(underserved_cluster + overserved_cluster)]
   .groupby("county")[div_cols + ["divergence"]].mean()
   .sort_values("divergence")
   .round(2))

,div_visits_per1k,div_individuals_per1k,div_pounds_per1k,divergence
county,,,,
Swift,-1.57,-1.50,-1.81,-1.63
Todd,-1.42,-1.32,-1.23,-1.32
Benton,-1.27,-1.23,-1.43,-1.31
Grant,-1.40,-1.36,-1.17,-1.31
Aitkin,-1.15,-1.31,-1.41,-1.29
Nicollet,-1.23,-1.07,-1.32,-1.21
Norman,-1.19,-1.05,-1.34,-1.19
Wadena,-1.14,-1.09,-1.14,-1.12
Chippewa,-1.01,-1.00,-1.18,-1.06


**Swift County is the cleanest single case:** one food shelf site for the entire county (`shelves = 1`, so `top_site_share = 1.0`), the most rural code in the dataset (`rucc = 9`, shared with Grant, Wadena and others), among the deepest poverty shares (`fi_below_snap` ~0.81–0.86), and the most negative divergence score of any county in 2023 and 2024 (third in 2022, behind Grant and Benton).

**Seven of the nine counties have three or fewer shelves.** Chippewa and Nicollet each have a single shelf in all three years, and Grant, Todd, Norman and Benton have 2–3. Most are rural, but the full-panel test in 4b shows that **rurality is the stronger and more consistent predictor**. Shelf density per food-insecure resident adds a smaller effect. A low shelf count can also overstate how thin a network is: Grant and Norman have only 2–3 shelves but 25–46 per 10k food-insecure residents, well above the panel median of 11.1. Swift, Todd, Benton, Chippewa and Nicollet (2.6–10.9 per 10k) are thin by both measures. Nicollet and Benton (`rucc = 3`) are the metro exceptions.

**Aitkin and Wadena are the exceptions.** They have thicker, less concentrated networks (Aitkin 6–9 shelves, `top_site_share` 0.29–0.47; Wadena 4 shelves, 0.35–0.38), but they still rank as under-served. Their divergence comes from the **need side**: deep poverty (`fi_below_snap` 0.71–0.84) and, in Aitkin's case, one of the highest meal costs in the state. This is a second route to the same outcome. Need can outrun activity because the network is thin, or because need is unusually deep even where the network isn't thin.

**Contrast with the over-served cluster** (Rice, Steele, Washington): lower rurality (`rucc` 1–5) and lower poverty depth (`fi_below_snap` 0.43–0.70). None of the three has a dense network per person. All are below the panel median of 11.1 shelves per 10k food-insecure residents (5.8–10.6), thinner than Grant, Norman, Aitkin or Wadena. Their high activity comes from large or regional sites and easier access, not from shelf density:
- **Steele** is the clearest case. One regional-catchment site accounts for 92–95% of the county's individuals served (`top_site_share` 0.92–0.95, table above), confirming Brian's Q1 finding. The divergence reflects one site drawing from beyond the county line, not genuine county-level need.
- **Washington** has the most shelves of any county shown (13), but per food-insecure resident its network is among the thinnest (5.8–7.5 per 10k). One large site still dominates (`top_site_share` 0.72–0.78) in this suburban metro county (`rucc = 1`).
- **Rice** is the least concentrated (5–7 shelves, `top_site_share` 0.33–0.49, 8.7–10.6 per 10k). Its high activity comes from a broad network rather than one hub, so it is the over-served case least explained by catchment.

**This is the throughline for the presentation:** rural counties, especially those with few food shelves, are structurally invisible to demand-based allocation, even though their need (by poverty depth, not just insecurity prevalence) is comparable to or higher than in counties that get far more visits. Swift's single shelf is doing the work an entire county's need would otherwise require. It shows up as "normal" in Q1's demand data because there's nothing to compare it *against* until you bring in a need measure Q1 didn't use.

### 4b. Does this hold across all 258 county-years, not just the two clusters?

Section 4 shows the contrast between two hand-picked clusters. To make the
"county characteristics explain divergence" claim rigorous — the way Q1's
regression made its claims rigorous — test it across the whole panel instead
of just the counties we chose to highlight.

In [34]:
import statsmodels.api as sm

char_corr = panel[["divergence", "rucc", "shelves_per_10k_fi", "fi_below_snap", "child_fi_rate"]].corr()["divergence"].drop("divergence")
print(char_corr.sort_values())

# standard errors clustered by county: each county appears 3 times (2022-24), so rows aren't independent
groups = panel["county"]

# full model
X_full = sm.add_constant(panel[["rucc", "shelves_per_10k_fi", "fi_below_snap", "child_fi_rate"]])
m_full = sm.OLS(panel["divergence"], X_full).fit(cov_type="cluster", cov_kwds={"groups": groups})
print(m_full.summary())

# clean model: only the access/structure characteristics, no need-index components on the right-hand side
X_clean = sm.add_constant(panel[["rucc", "shelves_per_10k_fi"]])
m_clean = sm.OLS(panel["divergence"], X_clean).fit(cov_type="cluster", cov_kwds={"groups": groups})
print(m_clean.summary())

# range of shelves_per_10k_fi, to translate the coefficient into a real-world difference
print(panel.shelves_per_10k_fi.describe())

fi_below_snap        -0.429127
rucc                 -0.312035
child_fi_rate        -0.273161
shelves_per_10k_fi    0.014764
Name: divergence, dtype: float64
                            OLS Regression Results                            
Dep. Variable:             divergence   R-squared:                       0.207
Model:                            OLS   Adj. R-squared:                  0.195
Method:                 Least Squares   F-statistic:                     5.607
Date:                Mon, 12 Oct 2026   Prob (F-statistic):           0.000470
Time:                        00:00:03   Log-Likelihood:                -349.65
No. Observations:                 258   AIC:                             709.3
Df Residuals:                     253   BIC:                             727.1
Df Model:                           4                                         
Covariance Type:              cluster                                         
                         coef    std err          z  

In [35]:
print(f"corr(child_fi_rate, fi_rate) = {panel[['child_fi_rate', 'fi_rate']].corr().iloc[0, 1]:.3f}")

corr(child_fi_rate, fi_rate) = 0.911


**Result (standard errors clustered by county, since each county appears in all three years):**

| | Full model | Clean model (no need-index components) |
|---|---|---|
| `rucc` (rurality, 1–9) | −0.058 (p = 0.120) | **−0.141 (p < 0.001)** |
| `shelves_per_10k_fi` | +0.012 (p = 0.060) | **+0.015 (p = 0.040)** |
| `fi_below_snap` | −3.69 (p = 0.005)* | not included |
| `child_fi_rate` | −0.67 (p = 0.818) | not included |
| R² | 0.207 | 0.124 |

\* Partly mechanical: `fi_below_snap` is one of the three components of `need_index`, so it sits on both sides of the equation. The clean model leaves it out so that only access and structure characteristics are tested.

**What this says:**
- **Rurality is the most robust county characteristic.** In the clean model, moving from the most urban county (`rucc` = 1) to the most rural (`rucc` = 9) predicts a divergence about 1.13 lower. That is roughly the gap between an average county and the under-served cluster in Section 4. In the full model, the rurality effect shrinks and loses significance once poverty depth is added. Rural counties also have deeper poverty, so part of the rurality effect works through poverty depth.
- **Shelf density per food-insecure resident matters, but less.** Fewer shelves per 10,000 food-insecure people go with more under-served divergence (p = 0.040 clean, p = 0.060 full). The effect is modest: moving from the 25th to the 75th percentile of shelf density (7.5 to 20.6 shelves per 10k) predicts divergence about 0.19 higher. The raw shelf counts in Section 4 overstate how thin these networks are, because many under-served counties are also small.
- **Child food insecurity adds nothing** once the other measures are in the model. This is consistent with leaving it out of the need index (r = 0.91 with `fi_rate`).
- **County characteristics explain only part of the divergence** (R² = 0.12–0.21). Most of the variation comes from things county-level data can't see, such as a single regional site drawing clients across county lines (Steele) or local outreach and awareness.

**Caveats:** the residuals are right-skewed (skew 1.68), driven by the large positive divergences of the over-served counties. The low Durbin-Watson (0.81) reflects the repeated-county structure that the clustered standard errors correct for. With 86 county clusters, p-values near 0.05 should be read as suggestive rather than conclusive.

### 4c. Is the finding robust to how the need index is built?

The index weights its three components equally, and that is a choice, not a law. If the under-served cluster only appears under this exact weighting, it is fragile. Below, the index is rebuilt five ways and two things are tracked for each version: how closely its divergence ranking matches the baseline (Spearman rank correlation), and how many of the nine Section 4 counties still rank in the bottom 10 in at least 2 of 3 years. The baseline row should reproduce 9 of 9 and serves as a check on the code. The `fi_rate only` row is Q1's view, so the counties it misses are the ones the richer index adds.

In [36]:
def build_need(cols, weights=None):
    z = panel[[f"z_{c}" for c in cols]]
    w = np.ones(len(cols)) if weights is None else np.asarray(weights, dtype=float)
    return (z * w).sum(axis=1) / w.sum()

variants = {
    "baseline (equal weights)":        build_need(NEED_COLS),
    "fi_rate only (Q1's view)":        build_need(["fi_rate"]),
    "drop cost_per_meal":              build_need(["fi_rate", "fi_below_snap"]),
    "drop fi_below_snap":              build_need(["fi_rate", "cost_per_meal"]),
    "fi_rate double-weighted (2:1:1)": build_need(NEED_COLS, [2, 1, 1]),
}

base_div = panel["demand_index"] - variants["baseline (equal weights)"]
cluster = set(underserved_cluster)
rows = []
for name, need_v in variants.items():
    div_v = panel["demand_index"] - need_v
    rank_v = div_v.groupby(panel["year"]).rank()
    bottom_counts = panel.loc[rank_v <= 10, "county"].value_counts()
    persistent = set(bottom_counts[bottom_counts >= 2].index)   # bottom 10 in >= 2 of 3 years
    rows.append({
        "variant": name,
        "spearman_vs_baseline": round(div_v.corr(base_div, method="spearman"), 3),
        "cluster_still_persistent (of 9)": len(persistent & cluster),
        "cluster_dropped": ", ".join(sorted(cluster - persistent)) or "-",
        "new_persistent": ", ".join(sorted(persistent - cluster)) or "-",
    })

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

,variant,spearman_vs_baseline,cluster_still_persistent (of 9),cluster_dropped,new_persistent
0,baseline (equal weights),1.000,9,-,-
1,fi_rate only (Q1's view),0.811,4,"Benton, Grant, Nicollet, Norman, Todd","Beltrami, Big Stone, Clearwater, Nobles"
2,drop cost_per_meal,0.894,6,"Benton, Nicollet, Norman","Clearwater, Lyon, Nobles"
3,drop fi_below_snap,0.898,7,"Grant, Wadena","Carver, Chisago, Fillmore"
4,fi_rate double-weighted (2:1:1),0.979,9,-,Nobles


In [37]:
# Wilkin check: does it really have zero food-shelf rows, while MMG still estimates need there?
print("Wilkin in raw fs:      ", fs.County.astype(str).str.contains("Wilkin", case=False).any())
print("Wilkin in cleaned fsq1:", fsq1.county.astype(str).str.contains("Wilkin", case=False).any())
print("Wilkin in MMG:         ", (mmg_s.county == "Wilkin").any())

# state median food-insecurity rate by year, to check "above the state median"
print("\nState median fi_rate by year:")
print(mmg_s.groupby("year").fi_rate.median())

mmg_s[mmg_s.county == "Wilkin"][["county", "year", "fi_persons", "fi_rate"]]

Wilkin in raw fs:       False
Wilkin in cleaned fsq1: False
Wilkin in MMG:          True

State median fi_rate by year:
year
2021    0.069
2022    0.100
2023    0.111
2024    0.123
Name: fi_rate, dtype: float64


,county,year,fi_persons,fi_rate
332,Wilkin,2021,510,0.078
333,Wilkin,2022,720,0.111
334,Wilkin,2023,790,0.124
335,Wilkin,2024,810,0.128


In [38]:
# share of statewide demand with no county assigned (excluded from every county-level number)
print((unassigned[M].sum() / fsq1[M].sum()).round(4))

HouseholdsReg     0.0110
IndividualsReg    0.0100
PoundsReg         0.0088
dtype: float64


## 5. Known caveats to state out loud

- **Wilkin County has zero rows in the raw data at all**: not in this panel, not in the window (see the check above). MMG puts 810 food-insecure residents there in 2024 (12.8% rate, above the 2024 state median of 12.3%), and its rate is above the state median in every year from 2021 to 2024. It is the single cleanest illustration of the Q6 premise, need with *no* demand signal whatsoever, but no regression here can score it. Report it as a named exception, not a model output.
- **Becker and Beltrami have under 35% household-figure coverage** (Q1 Decision 4). Their divergence scores in this notebook are real but less trustworthy than those of counties with ~100% coverage. Becker ranks in the bottom 10 once and is left out of the Section 4 cluster for this reason.
- **The unassigned residual** (`q1_unassigned_residual.csv`): about 1% of statewide demand (1.1% of households, 1.0% of individuals, 0.9% of pounds; see the check above) has no county at all. It is excluded from every county-level number here, same as Q1.
- **`shelves` counts sites that reported activity, not every food shelf that exists.** A county with shelves that didn't report to The Food Group will look thinner than it is. Cross-checking against the full `Organization.xlsx` roster would close this gap.
- **`top_site_share` is measured on individuals served** (`IndividualsReg`). On households or pounds, the concentration figures could differ somewhat.
- **County characteristics explain only part of the divergence** (R² = 0.12–0.21 in 4b). The rest comes from factors county-level data can't capture, such as cross-county catchment, outreach and awareness.

## 6. Graduate addendum — falsifiable definition (draft, needs team sign-off)

1. **Definition:** need = within-year z-score average of Map the Meal Gap's
   food-insecurity rate, cost-per-meal, and the share of food-insecure persons
   at or below the SNAP income threshold, by county-year.
2. **Why this is need and not demand:** none of the three components are built
   from The Food Group's own activity data — all three come from Feeding
   America's Map the Meal Gap, which models county food insecurity from national
   survey data (the Current Population Survey) and county economic indicators,
   independently of food shelf activity. Demand
   (`visits_per1k`, etc.) only enters at the comparison step.
3. **Evidence that would make us reject it:** if `need_index` turned out to be statistically indistinguishable from `fi_rate` alone (the other two components adding no new information), the "richer than demand" claim would collapse into "we renamed Q1's variable." **Tested; not rejected.** The correlation is r = 0.723, so `fi_rate` accounts for about 52% of the variation in `need_index` (r² = 0.523), and the other ~48% comes from cost burden and poverty depth. Neither added component duplicates `fi_rate`: `cost_per_meal` correlates at −0.55 with it and `fi_below_snap` at 0.58, far from the 0.91 that led us to leave out `child_fi_rate` (component correlations in Section 3; child rate in Section 1). The richer index also changes the outlier result: it newly flags **Cook** and clears **Ramsey**. The overlap is still large enough that we present the index as an extension of `fi_rate`, not a replacement for it.
4. **Evidence that would make us revise it:** (a) if The Food Group's own definition of "need" emphasizes a dimension we left out, especially access or distance; or (b) if an external data source (the brief notes that a complete measure "will likely require the use of other publicly available data") moves counties substantially in the ranking. On (a), Section 4b shows that rurality and shelf density behave as **explanations of divergence**, not as parts of need, which supports keeping them outside the index for now. If The Food Group defines need to include access, they would move inside it.

## Open questions for tomorrow's meeting

1. Keep `rucc` and `shelves_per_10k_fi` as *explanatory* variables only, or fold
   one into the need index itself (makes the index partly about access, not
   just underlying need — blurs the line with Q8)?
   **Update:** 4b supports keeping both explanatory. Rurality is the strongest
   predictor of divergence (p < 0.001), and shelf density has a smaller effect (p = 0.04).
2. The persistent under-served cluster is mostly small, rural counties — is a
   *rate-based* need index the right lens, or should we also show a *levels*
   version (total unmet need in raw people, not just rate) so Hennepin/Ramsey
   don't disappear from the conversation entirely?
3. Do we pull in one genuinely new external source (e.g., County Health
   Rankings access-to-care, or USDA Food Access Research Atlas distance
   measure) to strengthen "more than demand," or is the MMG-only version enough
   given the time left before Oct 17?
4. Who's drafting the falsifiable-definition slide language vs. the chart?
